# niftyhedge — Colab runbook

Follow **`RUNBOOK.md`**. Stop at every gate.

**Before this notebook:** on a **laptop in India**, probe one NSE UDiFF date, run the full download (Gate 1a), upload `repo/` + `data/raw/` to Drive at `MyDrive/Vol-Surfaces/`.

Runtime → Change runtime type → **GPU (T4)**.

## A. Confirm GPU

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"

## B. Mount Drive and set paths

Expected Drive layout:
```
MyDrive/Vol-Surfaces/
  repo/          # this repository
  data/raw/      # parquet + download_log.csv from laptop
  runs/          # created below
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from datetime import datetime

ROOT = "/content/drive/MyDrive/Vol-Surfaces"
REPO = f"{ROOT}/repo"
DATA_RAW = f"{ROOT}/data/raw"
RUN = f"{ROOT}/runs/{datetime.now().strftime('%Y%m%d')}"
os.makedirs(RUN, exist_ok=True)

os.environ["NH_ROOT"] = REPO
os.environ["NH_RUN"] = RUN
os.environ["NH_DATA_RAW"] = DATA_RAW

%cd {REPO}
!ls -la
!ls -la "{DATA_RAW}" | head

## C. Install package + link raw data

In [ ]:
!pip install -U pip -q
!pip install -e ".[dev]" -q

import os
dst = os.path.join(os.environ["NH_ROOT"], "data", "raw")
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.islink(dst) or os.path.exists(dst):
    if os.path.islink(dst):
        os.remove(dst)
    elif os.path.isdir(dst) and not os.listdir(dst):
        os.rmdir(dst)
if not os.path.exists(dst):
    os.symlink(os.environ["NH_DATA_RAW"], dst)
print("data/raw ->", os.path.realpath(dst))

!python -c "import niftyhedge, torch; print('niftyhedge OK', torch.cuda.is_available())"

## D. Gate 0 — build verification

Pass = all tests green, `check_build.py` A1–A13 PASS, `make smoke` exits 0.
**Stop if anything fails.**

In [ ]:
!pytest -q
!python scripts/check_build.py
!make smoke

## E. Legacy audit (documents the broken baseline)

Expected Normal ordering: BS +delta ≺ no hedge ≺ DDPM-RL ≺ BS −delta.

In [ ]:
!python legacy/audit_env.py . 300 | tee "$NH_RUN/legacy_audit.txt"

## F. Stage 1 — surfaces + folds (CPU)

Requires Gate 1a already passed on the laptop (`download_log.csv` clean).

In [ ]:
!python scripts/01_build_surfaces.py --config configs/data.yaml
!python scripts/02_make_folds.py --config configs/folds.yaml

import pandas as pd
print(pd.read_csv("tables/dataset_card.csv").to_string())

### Gate 1b checklist
- [ ] Last surface date ≥ 2026-03-01
- [ ] No NaN in `surfaces_v2.npy`
- [ ] Calendar + Durrleman violations < 5% of surfaces
- [ ] ATM 30d by year looks plausible (2020 highest)
- [ ] Extrapolated fraction reported; drop 7d/180d if >40% extrapolated

## G. Stage 2 — generators (GPU)

In [ ]:
!python scripts/10_fit_generators.py --config configs/gen/gbm.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/heston.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/pcavar.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/bootstrap.yaml --folds all --seeds 0
!python scripts/10_fit_generators.py --config configs/gen/ddpm.yaml --folds all --seeds 0,1,2
!python scripts/10_fit_generators.py --config configs/gen/ddpm.yaml --folds 5 --seeds 3,4

In [ ]:
!python scripts/11_generate_paths.py --worlds gbm,heston,pcavar,bootstrap,ddpm --folds all \
    --n-train 200000 --n-test 100000
!python scripts/12_eval_generators.py --config configs/experiments/e5_gen_quality.yaml

import pandas as pd
print(pd.read_csv("tables/gen_quality.csv").head(20).to_string())
print(pd.read_csv("tables/world_stats.csv").to_string())

### Gate 2 checklist
- [ ] DDPM CRPS < persistence on validation
- [ ] DDPM mean ATM 30d within 1 vol point of real
- [ ] Dose-response: ATM rises with lagged VIX
- [ ] 90% PI coverage in [80%, 97%]
- [ ] No NaN / non-positive vols in any world

## H. Gate 3 — W0 deep-hedge sanity (must pass before full training)

In [ ]:
!python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
    --world gbm --fold 5 --instruments I1 --contract configs/contracts/doc_090.yaml \
    --cost-bps 0 --seeds 0
!python scripts/check_build.py --only A14

## I. Stage 3 — train hedgers (GPU)

MVP shortcut: worlds `gbm,heston,pcavar,ddpm` and instruments `I1,I2` only.

In [ ]:
WORLDS = "gbm heston pcavar bootstrap ddpm"  # MVP: drop bootstrap
INSTRS = "I1 I2 I3"  # MVP: use "I1 I2"

for W in WORLDS.split():
    for I in INSTRS.split():
        !python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
            --world {W} --fold 5 --instruments {I} --contract configs/contracts/doc_090.yaml \
            --seeds 0,1,2,3,4
        !python scripts/20_train_hedgers.py --config configs/hedge/deep_hedge.yaml \
            --world {W} --folds 0,1,2,3,4 --instruments {I} --contract configs/contracts/doc_090.yaml \
            --seeds 0,1,2

!python scripts/20_train_hedgers.py --config configs/hedge/ppo.yaml \
    --world ddpm --fold 5 --instruments I1 --contract configs/contracts/doc_090.yaml \
    --seeds 0,1,2,3,4

## J. Stage 4 — evaluation + tables

In [ ]:
!python scripts/21_cross_world_eval.py --config configs/experiments/e7_cross_world.yaml
!python scripts/22_real_backtest.py --config configs/experiments/e8_real_backtest.yaml
!python scripts/30_sensitivity.py --config configs/experiments/e10_sensitivity.yaml
!python scripts/40_make_tables_figures.py --run "$NH_RUN"
!ls -la tables/ figures/ | head -40

### Gate 4 checklist
- [ ] Every test world: BS-BGK beats no hedge
- [ ] W0 zero-cost: learned hedger not better than BS-BGK beyond MC error
- [ ] Seed std reported for every learned hedger
- [ ] Real backtest states # starts, # non-overlapping windows, # knock-outs

Archive `tables/`, `figures/`, and `$NH_RUN/**/manifest.json` when done.

**Reconnect after disconnect:** remount Drive, reinstall (cells B–C), re-run the last unfinished command — completed seeds skip via `manifest.json`.